# Load data

## Import necessary libraries

In [410]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import scienceplots
# import seaborn as sns
import numpy as np
import json
import glob
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Set global variables

In [411]:
# Set style for better plots
# plt.style.use('seaborn-v0_8-paper')
# sns.set_palette("muted")

# Set style for paper-like plots
# plt.style.use(['science','ieee','no-latex'])
# plt.style.use(['science','no-latex'])
#plt.style.use(['science'])

# Global configuration for plot duration limit
# This is the maximum duration for which we will plot individual requests
# Necessary because malicious requests require longer to be interrupted in the tests
PLOT_DURATION_LIMIT = 60

## Load and preprocess the data

In [412]:
def load_latest_test_data(src_dir="./"):
    """Load the most recent test data"""
    # Look for both regular and malicious data files
    regular_files = glob.glob(f"{src_dir}/fairness_test_data_regular_*.csv")
    malicious_files = glob.glob(f"{src_dir}/fairness_test_data_malicious_*.csv")
    
    if not regular_files and not malicious_files:
        raise FileNotFoundError("No test data files found")
    
    # Get the latest timestamp
    all_files = regular_files + malicious_files
    latest_file = max(all_files)
    timestamp = latest_file.split('_')[-1].split('.')[0]
    
    # Load both regular and malicious data
    dfs = []
    
    regular_file = f"{src_dir}/fairness_test_data_regular_{timestamp}.csv"
    malicious_file = f"{src_dir}/fairness_test_data_malicious_{timestamp}.csv"
    
    if glob.glob(regular_file):
        df_regular = pd.read_csv(regular_file)
        dfs.append(df_regular)
    
    if glob.glob(malicious_file):
        df_malicious = pd.read_csv(malicious_file)
        dfs.append(df_malicious)
    
    if not dfs:
        raise FileNotFoundError(f"No data files found for timestamp {timestamp}")
    
    # Combine the dataframes
    df = pd.concat(dfs, ignore_index=True)
    
    # Load metadata
    metadata_file = f"{src_dir}/fairness_test_metadata_{timestamp}.json"
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    return df, metadata, timestamp

def load_baseline_test_data(src_dir="./"):
    """Load baseline test data (before stress test begins)"""
    # Look for baseline-specific files or use a different naming convention
    baseline_files = glob.glob(f"{src_dir}/fairness_baseline_data_*.csv")
    
    if not baseline_files:
        # If no specific baseline files, look for regular files with baseline timestamp
        baseline_files = glob.glob(f"{src_dir}/fairness_baseline_*.csv")
    
    if not baseline_files:
        raise FileNotFoundError("No baseline test data files found")
    
    # Get the latest baseline file
    latest_file = max(baseline_files)
    timestamp = latest_file.split('_')[-1].split('.')[0]
    
    # Load baseline data
    df_baseline = pd.read_csv(latest_file)
    
    # Load corresponding metadata
    metadata_file = f"{src_dir}/fairness_baseline_metadata_{timestamp}.json"
    if not glob.glob(metadata_file):
        # Fallback to regular metadata file
        metadata_file = f"{src_dir}/fairness_test_metadata_{timestamp}.json"
    
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    return df_baseline, metadata, timestamp

def preprocess_data(df, metadata):
    """Preprocess the individual request data"""
    # Convert start_time to numeric if it's not already
    df['start_time'] = pd.to_numeric(df['start_time'], errors='coerce')
    
    # Calculate elapsed seconds from the start of the test
    min_time = df['start_time'].min()
    df['elapsed_seconds'] = df['start_time'] - min_time
    
    # Convert duration from milliseconds to seconds for easier analysis
    df['duration_seconds'] = df['duration_ms'] / 1000.0
    
    # Calculate relative increase from baseline for each request
    baseline_latency = metadata['baseline_metrics']['tenant1_avg_latency_ms']
    df['relative_increase_percent'] = ((df['duration_ms'] - baseline_latency) / baseline_latency) * 100
    
    # Add phase identification based on the test type
    df['phase'] = 'stress'
    
    # Sort by time for proper plotting
    df = df.sort_values(['role', 'elapsed_seconds']).reset_index(drop=True)
    
    return df

def preprocess_baseline_data(df, metadata):
    """Preprocess baseline test data"""
    # Convert start_time to numeric if it's not already
    df['start_time'] = pd.to_numeric(df['start_time'], errors='coerce')
    
    # Calculate elapsed seconds from the start of the baseline test
    min_time = df['start_time'].min()
    df['elapsed_seconds'] = df['start_time'] - min_time
    
    # Convert duration from milliseconds to seconds for easier analysis
    df['duration_seconds'] = df['duration_ms'] / 1000.0
    
    # Add phase identification (all baseline data is 'baseline' phase)
    df['phase'] = 'baseline'
    
    # Sort by time for proper plotting
    df = df.sort_values(['role', 'elapsed_seconds']).reset_index(drop=True)
    
    return df


In [413]:
def save_plot(fig, filename):
    
    fig.savefig(f'{filename}.png', bbox_inches='tight', dpi=300)
    
    current_backend = matplotlib.get_backend()
    current_usetex = matplotlib.rcParams['text.usetex']
    
    # Switch to PGF for high-quality output
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "text.usetex": True,
        "pgf.rcfonts": False,
    })
    # Save without any axis title
    # Remove titles from all axes before saving
    for ax in fig.get_axes():
        ax.set_title('')  # Remove individual subplot titles
    
    fig.savefig(f'{filename}.pgf', bbox_inches='tight')
    
    # Switch back to interactive backend
    matplotlib.use(current_backend)
    matplotlib.rcParams.update({"text.usetex": current_usetex})


def plot_for_paper(text_width=3.31314,
                aspect_ratio=6/8,
                scale=1.0):
    """Resize the plot to a specific text width and aspect ratio"""
    scale = 1.0
    width = text_width * scale
    height = width * aspect_ratio
    fig, ax =  plt.subplots(figsize=(width, height), dpi=300)
    return fig,ax


## Plot functions

In [414]:

def aggregate_data_by_time_window(df, window_size=1.0):
    """Aggregate individual requests into time windows for analysis"""
    aggregated_data = []
    
    for role in df['role'].unique():
        role_data = df[df['role'] == role].copy()
        if len(role_data) == 0:
            continue
        
        # Create time bins
        max_time = role_data['elapsed_seconds'].max()
        time_bins = np.arange(0, max_time + window_size, window_size)
        
        for i in range(len(time_bins) - 1):
            bin_start = time_bins[i]
            bin_end = time_bins[i + 1]
            
            # Find all requests within this time bin
            bin_mask = (role_data['elapsed_seconds'] >= bin_start) & (role_data['elapsed_seconds'] < bin_end)
            bin_data = role_data[bin_mask]
            
            if len(bin_data) > 0:
                # Calculate aggregated metrics for this window
                avg_latency = bin_data['duration_ms'].mean()
                std_latency = bin_data['duration_ms'].std()
                error_rate = bin_data['is_error'].mean()
                total_requests = len(bin_data)
                
                aggregated_data.append({
                    'time_window_start': bin_start,
                    'time_window_end': bin_end,
                    'time_window_center': (bin_start + bin_end) / 2,
                    'role': role,
                    'avg_latency_ms': avg_latency,
                    'std_latency_ms': std_latency,
                    'error_rate': error_rate,
                    'total_requests': total_requests,
                    'requests_per_second': total_requests / window_size
                })
    
    return pd.DataFrame(aggregated_data)

def plot_individual_requests_scatter(df, metadata, timestamp, save_plots=True, out_dir="./"):   
    """Plot individual request latencies as scatter plot"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: Individual request latencies
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) == 0:
            continue
        
        # Use different markers for successful vs failed requests
        success_data = role_data[~role_data['is_error']]
        error_data = role_data[role_data['is_error']]
        
        if len(success_data) > 0:
            ax1.scatter(success_data['elapsed_seconds'], success_data['duration_ms'], 
                       label=f'{role} Tenant (Success)', alpha=0.6, s=10)
        
        if len(error_data) > 0:
            ax1.scatter(error_data['elapsed_seconds'], error_data['duration_ms'], 
                       label=f'{role} Tenant (Error)', alpha=0.8, s=15, marker='x')
    
    # Add baseline line
    baseline = metadata['baseline_metrics']['tenant1_avg_latency_ms']
    ax1.axhline(y=baseline,  linestyle='--', alpha=0.7, 
                label=f'Baseline ({baseline:.1f}ms)')
    
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Request Latency (ms)')
    ax1.set_title('Individual Request Latencies Over Time')
    ax1.set_xlim(0, PLOT_DURATION_LIMIT)
    ax1.legend()
    
    # Plot 2: Moving average with confidence intervals
    window_size = 1.0  # 1-second moving window
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) == 0:
            continue
        
        # Calculate rolling statistics
        role_data_sorted = role_data.sort_values('elapsed_seconds')
        
        # Create time bins for rolling average
        max_time = role_data_sorted['elapsed_seconds'].max()
        time_points = np.arange(0, max_time, 1.0)  # Every second
        
        rolling_mean = []
        rolling_std = []
        valid_time_points = []
        
        for t in time_points:
            # Get data within window_size/2 around this time point
            window_mask = (role_data_sorted['elapsed_seconds'] >= t - window_size/2) & \
                         (role_data_sorted['elapsed_seconds'] <= t + window_size/2)
            window_data = role_data_sorted[window_mask]
            
            #if len(window_data) >= 5:  # Only if we have enough data points
            rolling_mean.append(window_data['duration_ms'].mean())
            rolling_std.append(window_data['duration_ms'].std())
            valid_time_points.append(t)
        
        if rolling_mean:
            rolling_mean = np.array(rolling_mean)
            rolling_std = np.array(rolling_std)
            valid_time_points = np.array(valid_time_points)
            
            ax2.plot(valid_time_points, rolling_mean, label=f'{role} Tenant (Moving Avg)', linewidth=2)
            ax2.fill_between(valid_time_points, 
                           rolling_mean - rolling_std, 
                           rolling_mean + rolling_std, 
                           alpha=0.2)
    
    ax2.axhline(y=baseline,  linestyle='--', alpha=0.7, 
                label=f'Baseline ({baseline:.1f}ms)')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Average Latency (ms)')
    ax2.set_ylim(bottom=0)
    ax2.set_title(f'Moving Average Latency ({window_size}s window)')
    ax2.set_xlim(0, PLOT_DURATION_LIMIT)
    ax2.legend()
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(f'{out_dir}/individual_requests_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
def plot_relative_increase(df, metadata, timestamp, save_plots=True, out_dir="./"):
    """Plot relative increase in latency compared to baseline as a moving average"""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Calculate rolling average of relative increase
    window_size = 1.0  # 1-second moving window
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) == 0:
            continue
        
        # Sort by elapsed time
        role_data_sorted = role_data.sort_values('elapsed_seconds')
        
        # Create time bins for rolling average
        max_time = role_data_sorted['elapsed_seconds'].max()
        time_points = np.arange(0, max_time, 1.0)  # Every second
        
        rolling_mean = []
        rolling_std = []
        valid_time_points = []
        
        for t in time_points:
            # Get data within window_size/2 around this time point
            window_mask = (role_data_sorted['elapsed_seconds'] >= t - window_size/2) & \
                         (role_data_sorted['elapsed_seconds'] <= t + window_size/2)
            window_data = role_data_sorted[window_mask]
            
            if len(window_data) >= 3:  # Only if we have enough data points
                rolling_mean.append(window_data['relative_increase_percent'].mean())
                rolling_std.append(window_data['relative_increase_percent'].std())
                valid_time_points.append(t)
        
        if rolling_mean:
            rolling_mean = np.array(rolling_mean)
            rolling_std = np.array(rolling_std)
            valid_time_points = np.array(valid_time_points)
            
            ax.plot(valid_time_points, rolling_mean, label=f'{role} Tenant', linewidth=2)
            ax.fill_between(valid_time_points, 
                           rolling_mean - rolling_std, 
                           rolling_mean + rolling_std, 
                           alpha=0.2)
    
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Relative Increase (%)')
    ax.set_ylim(bottom=0)
    ax.set_title('Average Relative Latency Increase from a non contended environment (1s window)')
    ax.set_xlim(0, PLOT_DURATION_LIMIT)
    ax.legend()
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(f'{out_dir}/relative_increase_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_error_analysis(df, metadata, timestamp, save_plots=True, out_dir="./"):
    """Plot error analysis for individual requests"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. Error rate over time (aggregated by windows)
    aggregated_df = aggregate_data_by_time_window(df, window_size=1.0)
    
    for role in aggregated_df['role'].unique():
        role_data = aggregated_df[aggregated_df['role'] == role]
        if len(role_data) == 0:
            continue
        ax1.plot(role_data['time_window_center'], role_data['error_rate'] * 100, 
                label=f'{role} Tenant', marker='o', markersize=4)
    
    ax1.axhline(y=5,  linestyle='--', alpha=0.7, label='Failure Threshold (5%)')
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Error Rate (%)')
    ax1.set_title('Error Rate Over Time (1s windows)')
    ax1.set_xlim(0, PLOT_DURATION_LIMIT)
    ax1.legend()
    
    # 2. Error distribution by operation type
    error_by_operation = df.groupby(['role', 'operation'])['is_error'].agg(['count', 'sum', 'mean']).reset_index()
    error_by_operation['error_rate'] = error_by_operation['mean'] * 100
    
    operations = error_by_operation['operation'].unique()
    x_pos = np.arange(len(operations))
    width = 0.35
    
    for i, role in enumerate(error_by_operation['role'].unique()):
        role_data = error_by_operation[error_by_operation['role'] == role]
        values = [role_data[role_data['operation'] == op]['error_rate'].iloc[0] 
                 if len(role_data[role_data['operation'] == op]) > 0 else 0 
                 for op in operations]
        ax2.bar(x_pos + i * width, values, width, label=f'{role} Tenant')
    
    ax2.set_xlabel('Operation Type')
    ax2.set_ylabel('Error Rate (%)')
    ax2.set_title('Error Rate by Operation Type')
    ax2.set_xticks(x_pos + width / 2)
    ax2.set_xticklabels(operations)
    ax2.legend()
    
    # 3. Latency distribution (box plot)
    latency_data = []
    labels = []
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) > 0:
            latency_data.append(role_data['duration_ms'].values)
            labels.append(f'{role}\n(n={len(role_data)})')
    
    if latency_data:
        ax3.boxplot(latency_data, labels=labels, showfliers=False)
    ax3.set_ylabel('Latency (ms)')
    ax3.set_title('Latency Distribution by Role')
    
    # 4. Request throughput over time
    for role in aggregated_df['role'].unique():
        role_data = aggregated_df[aggregated_df['role'] == role]
        if len(role_data) == 0:
            continue
        ax4.plot(role_data['time_window_center'], role_data['requests_per_second'], 
                label=f'{role} Tenant', marker='o', markersize=4)
    
    ax4.set_xlabel('Time (seconds)')
    ax4.set_ylabel('Requests per Second')
    ax4.set_title('Request Throughput Over Time')
    ax4.set_xlim(0, PLOT_DURATION_LIMIT)
    ax4.legend()
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(f'{out_dir}/error_analysis_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_summary_dashboard(df, metadata, timestamp, save_plots=True, out_dir="./"):
    """Create a comprehensive dashboard for individual request data"""
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)
    
    # 1. Moving average latency (top row, full width)
    ax1 = fig.add_subplot(gs[0, :])
    window_size = 1.0
    for role in df['role'].unique():
        role_data = df[df['role'] == role].sort_values('elapsed_seconds')
        if len(role_data) == 0:
            continue
        
        # Calculate rolling average
        max_time = role_data['elapsed_seconds'].max()
        time_points = np.arange(0, max_time, 1.0)
        rolling_mean = []
        valid_time_points = []
        
        for t in time_points:
            window_mask = (role_data['elapsed_seconds'] >= t - window_size/2) & \
                         (role_data['elapsed_seconds'] <= t + window_size/2)
            window_data = role_data[window_mask]
            
            if len(window_data) >= 3:
                rolling_mean.append(window_data['duration_ms'].mean())
                valid_time_points.append(t)
        
        if rolling_mean:
            ax1.plot(valid_time_points, rolling_mean, label=f'{role} Tenant', linewidth=2)
    
    baseline = metadata['baseline_metrics']['tenant1_avg_latency_ms']
    ax1.axhline(y=baseline,  linestyle='--', alpha=0.7, 
                label=f'Baseline ({baseline:.1f}ms)')
    ax1.set_title('Average Request Latency Over Time')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_xlim(0, PLOT_DURATION_LIMIT)
    ax1.legend()
    
    # 2. Error rates (second row, left)
    ax2 = fig.add_subplot(gs[1, 0])
    aggregated_df = aggregate_data_by_time_window(df, window_size=1.0)
    for role in aggregated_df['role'].unique():
        role_data = aggregated_df[aggregated_df['role'] == role]
        if len(role_data) == 0:
            continue
        ax2.plot(role_data['time_window_center'], role_data['error_rate'] * 100, 
                label=f'{role}', marker='o', markersize=3)
    ax2.axhline(y=5,  linestyle='--', alpha=0.7, label='Threshold')
    ax2.set_title('Error Rate Over Time')
    ax2.set_ylabel('Error Rate (%)')
    ax2.legend()
    
    # 3. Latency distribution (second row, middle)
    ax3 = fig.add_subplot(gs[1, 1])
    latency_data = []
    labels = []
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) > 0:
            latency_data.append(role_data['duration_ms'].values)
            labels.append(role)
    
    if latency_data:
        ax3.boxplot(latency_data, labels=labels, showfliers=False)
    ax3.set_title('Latency Distribution')
    ax3.set_ylabel('Latency (ms)')
    
    # 4. Request throughput (second row, right)
    ax4 = fig.add_subplot(gs[1, 2])
    for role in aggregated_df['role'].unique():
        role_data = aggregated_df[aggregated_df['role'] == role]
        if len(role_data) == 0:
            continue
        ax4.plot(role_data['time_window_center'], role_data['requests_per_second'], 
                label=f'{role}', marker='o', markersize=3)
    ax4.set_title('Throughput Over Time')
    ax4.set_ylabel('Requests/sec')
    ax4.set_xlim(0, PLOT_DURATION_LIMIT)
    ax4.legend()
    
    # 5. Operation type analysis (third row, left and middle)
    ax5 = fig.add_subplot(gs[2, :2])
    operation_stats = df.groupby(['role', 'operation']).agg({
        'duration_ms': ['count', 'mean', 'std'],
        'is_error': 'mean'
    }).round(2)
    
    # Flatten column names
    operation_stats.columns = ['_'.join(col).strip() for col in operation_stats.columns]
    operation_stats = operation_stats.reset_index()
    
    # Create a grouped bar chart for average latency by operation
    operations = operation_stats['operation'].unique()
    x_pos = np.arange(len(operations))
    width = 0.35
    
    for i, role in enumerate(operation_stats['role'].unique()):
        role_data = operation_stats[operation_stats['role'] == role]
        values = [role_data[role_data['operation'] == op]['duration_ms_mean'].iloc[0] 
                 if len(role_data[role_data['operation'] == op]) > 0 else 0 
                 for op in operations]
        ax5.bar(x_pos + i * width, values, width, label=f'{role} Tenant')
    
    ax5.set_title('Average Latency by Operation Type')
    ax5.set_ylabel('Average Latency (ms)')
    ax5.set_xticks(x_pos + width / 2)
    ax5.set_xticklabels(operations)
    ax5.legend()
    
    # 6. Resource type analysis (third row, right)
    ax6 = fig.add_subplot(gs[2, 2])
    resource_stats = df.groupby(['role', 'resource']).agg({
        'duration_ms': 'mean',
        'is_error': 'mean'
    }).reset_index()
    
    resources = resource_stats['resource'].unique()
    x_pos = np.arange(len(resources))
    
    for i, role in enumerate(resource_stats['role'].unique()):
        role_data = resource_stats[resource_stats['role'] == role]
        values = [role_data[role_data['resource'] == res]['duration_ms'].iloc[0] 
                 if len(role_data[role_data['resource'] == res]) > 0 else 0 
                 for res in resources]
        ax6.bar(x_pos + i * width, values, width, label=f'{role}')
    
    ax6.set_title('Avg Latency by Resource')
    ax6.set_ylabel('Latency (ms)')
    ax6.set_xticks(x_pos + width / 2)
    ax6.set_xticklabels(resources)
    ax6.legend()
    
    # 7. Summary statistics table (bottom row)
    ax7 = fig.add_subplot(gs[3, :])
    ax7.axis('tight')
    ax7.axis('off')
    
    summary_data = []
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) == 0:
            continue
            
        avg_latency = role_data['duration_ms'].mean()
        median_latency = role_data['duration_ms'].median()
        p95_latency = role_data['duration_ms'].quantile(0.95)
        std_latency = role_data['duration_ms'].std()
        error_rate = role_data['is_error'].mean()
        total_requests = len(role_data)
        
        summary_data.append([
            role,
            f"{total_requests:,}",
            f"{avg_latency:.1f}ms",
            f"{median_latency:.1f}ms", 
            f"{p95_latency:.1f}ms",
            f"{std_latency:.1f}ms",
            f"{error_rate * 100:.2f}%"
        ])
    
    if summary_data:
        table = ax7.table(cellText=summary_data,
                          colLabels=['Tenant', 'Total Requests', 'Avg Latency', 
                                   'Median Latency', 'P95 Latency', 'Std Dev', 'Error Rate'],
                          cellLoc='center',
                          loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2)
    ax7.set_title('Individual Request Statistics Summary', pad=20)
    
    # Add test result
    test_result = "PASSED" if metadata.get('test_passed', False) else "FAILED"
    fig.suptitle(f'Fairness Test Analysis - Individual Requests - {test_result}', 
                 fontsize=16, fontweight='bold')
    
    if save_plots:
        plt.savefig(f'{out_dir}/individual_requests_dashboard_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()

def print_test_summary(df, metadata):
    """Print a detailed test summary for individual request data"""
    print("=" * 70)
    print("FAIRNESS TEST SUMMARY - INDIVIDUAL REQUEST ANALYSIS")
    print("=" * 70)
    
    config = metadata.get('test_config', {})
    baseline = metadata.get('baseline_metrics', {})
    final = metadata.get('final_results', {})
    
    print(f"Test Result: {'✅ PASSED' if metadata.get('test_passed', False) else '❌ FAILED'}")
    print()
    
    print("Test Configuration:")
    print(f"  Regular Requesters: {config.get('regular_requesters', 'N/A')}")
    print(f"  Malicious Requesters: {config.get('malicious_requesters', 'N/A')}")
    print(f"  Regular Request Rate: {config.get('regular_request_rate', 'N/A')}/s")
    print(f"  Malicious Request Rate: {config.get('malicious_request_rate', 'N/A')}/s")
    print(f"  Test Duration: {config.get('test_duration', {}).get('secs', 'N/A')}s")
    print()
    
    print("Individual Request Statistics:")
    for role in df['role'].unique():
        role_data = df[df['role'] == role]
        if len(role_data) == 0:
            continue
        
        print(f"\n  {role} Tenant:")
        print(f"    Total Requests: {len(role_data):,}")
        print(f"    Average Latency: {role_data['duration_ms'].mean():.1f}ms")
        print(f"    Median Latency: {role_data['duration_ms'].median():.1f}ms")
        print(f"    95th Percentile: {role_data['duration_ms'].quantile(0.95):.1f}ms")
        print(f"    Standard Deviation: {role_data['duration_ms'].std():.1f}ms")
        print(f"    Error Rate: {role_data['is_error'].mean() * 100:.2f}%")
        print(f"    Successful Requests: {(~role_data['is_error']).sum():,}")
        print(f"    Failed Requests: {role_data['is_error'].sum():,}")
    
    print(f"\nBaseline Comparison:")
    baseline_latency = baseline.get('tenant1_avg_latency_ms', 0)
    regular_data = df[df['role'] == 'Regular']
    if len(regular_data) > 0:
        current_avg = regular_data['duration_ms'].mean()
        increase = ((current_avg - baseline_latency) / baseline_latency * 100) if baseline_latency > 0 else 0
        print(f"  Baseline Latency: {baseline_latency:.1f}ms")
        print(f"  Current Avg Latency: {current_avg:.1f}ms")
        print(f"  Relative Increase: {increase:.1f}%")
        
def plot_baseline_to_stress_transition(df_baseline, df_stress, metadata, timestamp, save_plots=True, out_dir="./"):
    """Plot a single timeline showing baseline phase (0-10s) followed by stress test phase"""
    
    # Merge the dataframes with proper time offset
    baseline_duration = 10  # Fixed 10 seconds for baseline
    
    # Prepare baseline data (0-10 seconds)
    df_baseline_copy = df_baseline.copy()
    # Ensure baseline data stays within 0-10 seconds
    df_baseline_copy = df_baseline_copy[df_baseline_copy['elapsed_seconds'] <= baseline_duration]
     
    # Prepare stress data (10+ seconds) 
    df_stress_copy = df_stress.copy()
    # Offset stress test data to start at 10 seconds
    df_stress_copy['elapsed_seconds'] = df_stress_copy['elapsed_seconds'] + baseline_duration
    
    # Merge both dataframes
    df_combined = pd.concat([df_baseline_copy, df_stress_copy], ignore_index=True)
    df_combined = df_combined.sort_values('elapsed_seconds').reset_index(drop=True)
    
    # Get test configuration
    baseline_latency = metadata['baseline_metrics']['tenant1_avg_latency_ms']
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    window_size = 2  # 1-second moving window
    window_step = 0.125  # Step size for rolling mean calculation
    
    print(f"Plotting combined timeline with {len(df_combined)} total requests")
    print(f"Baseline phase: 0-{baseline_duration}s, Stress phase: {baseline_duration}s+")
    
    line_width = 2.5
    # Take colors from the scienceplots style
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    # Use a color map for different roles
    color_map = {
        'Regular': colors[0],  # First color for Regular
        'Malicious': colors[1],  # Second color for Malicious
    }
    
    # Plot each role using the combined dataframe
    for role in df_combined['role'].unique():
        role_data = df_combined[df_combined['role'] == role].sort_values('elapsed_seconds')
        if len(role_data) == 0:
            continue
        
        # Separate baseline and stress portions for different styling
        baseline_portion = role_data[role_data['elapsed_seconds'] <= baseline_duration]
        stress_portion = role_data[role_data['elapsed_seconds'] > baseline_duration]
        
        baseline_times = []
        baseline_values = []
        stress_times = []
        stress_values = []
        
        # Plot baseline portion
        if len(baseline_portion) > 0 and role == 'Regular':
            time_points = np.arange(0, baseline_duration, window_step)
            rolling_mean = []
            rolling_std = []
            valid_time_points = []
            
            for t in time_points:
                window_mask = (baseline_portion['elapsed_seconds'] >= t - window_size/2) & \
                             (baseline_portion['elapsed_seconds'] <= t + window_size/2)
                window_data = baseline_portion[window_mask]
                
                if len(window_data) >= 1:
                    rolling_mean.append(window_data['duration_ms'].mean())
                    rolling_std.append(window_data['duration_ms'].std())
                    valid_time_points.append(t)
            
            if rolling_mean:
                baseline_times = valid_time_points
                baseline_values = rolling_mean
                rolling_mean = np.array(rolling_mean)
                rolling_std = np.array(rolling_std)
                valid_time_points = np.array(valid_time_points)

                ax.plot(valid_time_points, rolling_mean, 
                       label=None,  # This should be not duplicated with the stress phase
                       linewidth=2.5, alpha=0.9, color=color_map[role])
                ax.fill_between(valid_time_points, 
                               np.maximum(rolling_mean - rolling_std, 0),
                               rolling_mean + rolling_std, 
                               alpha=0.2, color=color_map[role])
                                
        
        # Plot stress portion (solid lines)
        if len(stress_portion) > 0:
            max_stress_time = min(stress_portion['elapsed_seconds'].max(), PLOT_DURATION_LIMIT)
            time_points = np.arange(baseline_duration, max_stress_time, window_step)
            rolling_mean = []
            rolling_std = []
            valid_time_points = []
            
            for t in time_points:
                window_mask = (stress_portion['elapsed_seconds'] >= t - window_size/2) & \
                             (stress_portion['elapsed_seconds'] <= t + window_size/2)
                window_data = stress_portion[window_mask]
                
                if len(window_data) >= 3:
                    rolling_mean.append(window_data['duration_ms'].mean())
                    rolling_std.append(window_data['duration_ms'].std())
                    valid_time_points.append(t)
            
            if rolling_mean:
                stress_times = valid_time_points
                stress_values = rolling_mean
                rolling_mean = np.array(rolling_mean)
                rolling_std = np.array(rolling_std)
                valid_time_points = np.array(valid_time_points)
                
                # Different styling for stress phase
                line_style = '-' if role == 'Regular' else '-'
                
                ax.plot(valid_time_points, rolling_mean, 
                       label=f'{role} Tenant', 
                       linewidth=line_width, linestyle=line_style, color=color_map[role])
                
                # Add confidence intervals for stress phase
                ax.fill_between(valid_time_points, 
                               np.maximum(rolling_mean - rolling_std, 0), 
                               rolling_mean + rolling_std, 
                               alpha=0.2, color=color_map[role])
                
        # ADD BRIDGE CONNECTION
        # if len(baseline_times) > 0 and len(stress_times) > 0:
        #     # Connect last baseline point to first stress point
        #     last_baseline_time = baseline_times[-1]
        #     last_baseline_value = baseline_values[-1]
        #     first_stress_time = stress_times[0]
        #     first_stress_value = stress_values[0]
            
        #     # Create bridge points
        #     bridge_times = [last_baseline_time, baseline_duration, first_stress_time]
        #     bridge_values = [last_baseline_value, last_baseline_value, first_stress_value]
            
        #     color = 'blue' if role == 'Regular' else 'red'
        #     ax.plot(bridge_times, bridge_values, 
        #            color=color, linewidth=2.5, linestyle='-')
    
    # Add visual elements
    ax.axhline(y=baseline_latency, color='gray', linestyle='--', alpha=0.7, 
               label=None)
               #label=f'Baseline Reference ({baseline_latency:.1f}ms)')
    
    ax.axvline(x=baseline_duration, color='black', linestyle='--', alpha=1, linewidth=1.5,
               label='Malicious Tenant Introduced')
    
    #ax.axvspan(0, baseline_duration, alpha=0.15, color='green', label='Baseline Phase')
    #ax.axvspan(baseline_duration, PLOT_DURATION_LIMIT, alpha=0.15, color='orange', label='Stress Test Phase')
    
    # Customize the plot
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Request Latency (ms)')
    ax.set_title('Latency from Baseline to the Stress Phase')
    ax.set_xlim(0, PLOT_DURATION_LIMIT)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    #log scale
    ax.set_yscale('log')
    ax.set_ylim(bottom=1)  # Avoid log(0) issues
    
    
    
    plt.tight_layout()
    
    if save_plots:
        plt.savefig(f'{out_dir}/baseline_to_stress_transition_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics using the combined dataframe
    print("\n" + "="*60)
    print("BASELINE TO STRESS TRANSITION ANALYSIS")
    print("="*60)
    
    baseline_stats = {}
    baseline_data = df_combined[df_combined['elapsed_seconds'] <= baseline_duration]
    stress_data = df_combined[df_combined['elapsed_seconds'] > baseline_duration]
    
    print(f"\nBASELINE PHASE (0-{baseline_duration}s):")
    for role in baseline_data['role'].unique():
        role_data = baseline_data[baseline_data['role'] == role]
        if len(role_data) > 0:
            avg_latency = role_data['duration_ms'].mean()
            baseline_stats[role] = avg_latency
            print(f"  {role} Tenant: {avg_latency:.1f}ms avg, {len(role_data)} requests")
    
    print(f"\nSTRESS TEST PHASE ({baseline_duration}s-{PLOT_DURATION_LIMIT}s):")
    for role in stress_data['role'].unique():
        role_data = stress_data[stress_data['role'] == role]
        if len(role_data) > 0:
            avg_latency = role_data['duration_ms'].mean()
            baseline_avg = baseline_stats.get(role, baseline_latency)
            increase_pct = ((avg_latency - baseline_avg) / baseline_avg * 100) if baseline_avg > 0 else 0
            print(f"  {role} Tenant: {avg_latency:.1f}ms avg (+{increase_pct:.1f}%), {len(role_data)} requests")
    
    # Calculate fairness impact
    regular_baseline = baseline_stats.get('Regular', baseline_latency)
    regular_stress_data = stress_data[stress_data['role'] == 'Regular']
    if len(regular_stress_data) > 0:
        regular_stress_avg = regular_stress_data['duration_ms'].mean()
        impact = ((regular_stress_avg - regular_baseline) / regular_baseline * 100)
        print(f"\nFAIRNESS IMPACT:")
        print(f"  Regular tenant degradation: {impact:.1f}%")
        print(f"  {'✅ Fair' if impact < 50 else '❌ Unfair'} (threshold: 50%)")

# Main analysis function
def analyze_fairness_test(src_dir="./", out_dir="./"):
    """Run complete analysis for individual request data"""
    try:
        # Load baseline data
        df_baseline, baseline_metadata, baseline_timestamp = load_baseline_test_data(src_dir)
        df_baseline = preprocess_baseline_data(df_baseline, baseline_metadata)
        
        # Load stress test data
        df_stress, stress_metadata, stress_timestamp = load_latest_test_data(src_dir)
        df_stress = preprocess_data(df_stress, stress_metadata)
        
        print(f"Loaded {len(df_baseline)} baseline request records")
        print(f"Loaded {len(df_stress)} stress test request records")
        
        # Use stress test metadata as it contains the full test information
        metadata = stress_metadata
        timestamp = f"{baseline_timestamp}_{stress_timestamp}"
        
        print_test_summary(df_stress, metadata)
        print("\nGenerating baseline vs stress comparison plot...")
        
        # Generate the comparison plot
        plot_baseline_to_stress_transition(df_baseline, df_stress, metadata, timestamp, save_plots=True, out_dir=out_dir)
        
        # You can still generate other plots using stress data
        plot_individual_requests_scatter(df_stress, metadata, stress_timestamp, save_plots=True, out_dir=out_dir)
        plot_relative_increase(df_stress, metadata, stress_timestamp, save_plots=True, out_dir=out_dir)
        plot_error_analysis(df_stress, metadata, stress_timestamp, save_plots=True, out_dir=out_dir)
        plot_summary_dashboard(df_stress, metadata, stress_timestamp, save_plots=True, out_dir=out_dir)
        
        print(f"\nAnalysis complete! Plots saved with timestamp: {timestamp}")
        
        return df_baseline, df_stress, metadata
 
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        import traceback
        traceback.print_exc()
        return None, None

In [ ]:
def plot_comparative_latency_distribution(data_dirs, save_plots=True, out_dir="./", single_column_size=True):
    """Create a comparative box plot showing latency distribution by role for all solutions"""
    
    solutions = ['capsule', 'kubezoo', 'vcluster', 'kubevirt']
    solutions_label = ['Capsule', 'KubeZoo', 'vCluster', 'KubeVirt']
    all_data = {}
    
    # Load data from all solutions
    for solution in solutions:
        try:
            solution_dir = f"{data_dirs}/{solution}/data"
            df, metadata, timestamp = load_latest_test_data(solution_dir)
            df = preprocess_data(df, metadata)
            all_data[solution] = df
            print(f"Loaded {len(df)} records for {solution}")
        except Exception as e:
            print(f"Warning: Could not load data for {solution}: {e}")
            continue
    
    if not all_data:
        print("No data loaded for any solution")
        return
    
    fig, ax = None, None
    # Create the plot
    if single_column_size:
        fig, ax = plot_for_paper()
    else:
        fig, ax = plot_for_paper(text_width=6.6)
    
    # Prepare data for box plot
    box_data = []
    box_labels = []
    
    # Get default color cycle from the current style
    prop_cycle = plt.rcParams['axes.prop_cycle']
    colors = prop_cycle.by_key()['color']
    
    # Use first two colors from the style for Regular and Malicious
    role_colors = {
        'Regular': colors[0],    # First color from style
        'Malicious': colors[1]   # Second color from style
    }
    
    positions = []
    pos = 1
    solution_positions = []
    solution_labels = []

    for (solution, label) in zip(solutions, solutions_label):
        if solution not in all_data:
            continue
            
        df = all_data[solution]
        solution_start_pos = pos
        
        for role in ['Regular', 'Malicious']:
            role_data = df[df['role'] == role]
            if len(role_data) > 0:
                latencies = role_data['duration_ms']
                box_data.append(latencies.values)
                box_labels.append(role)  # Just the role name
                positions.append(pos)
                pos += 0.3
        
        # Store solution position (center between the two boxes)
        solution_positions.append((solution_start_pos + pos - 0.3) / 2)
        solution_labels.append(label)
        
        # Add spacing between solutions
        pos += 0.6

    
    # Create box plot
    bp = ax.boxplot(box_data, positions=positions, patch_artist=True, 
                    labels=box_labels, showfliers=False,
                    widths=0.3,
                    medianprops={'color': 'black'})
    
    import matplotlib.colors as mcolors
    # Color the boxes
    for patch, label in zip(bp['boxes'], box_labels):
        patch.set_facecolor(role_colors[label])
        patch.set_alpha(0.6)  # Set transparency for better visibility
        darker_color = mcolors.to_rgb(role_colors[label])
        darker_color = tuple(c * 0.6 for c in darker_color)
        patch.set_edgecolor(darker_color)
        patch.set_linewidth(1)
        
    
    # Customize the plot
    ax.set_ylabel('Request Latency (ms)')
    ax.set_title('Latency Distribution across Solutions')
    
    # Set custom x-axis labels
    ax.set_xticks(solution_positions)
    ax.set_xticklabels(solution_labels)
    
    # Add role labels as secondary x-axis
    # for i, pos in enumerate(positions):
    #     role = box_labels[i]
    #     ax.text(pos, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.03, 
    #         role, ha='center', va='top', fontsize=8)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=role_colors['Regular'], alpha=0.7, label='Regular'),
        Patch(facecolor=role_colors['Malicious'], alpha=0.7, label='Malicious')
    ]
    
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1, 1), handletextpad=0.25)
    # Reduce x-axis padding
    ax.set_xlim(0.5, max(positions) + 0.5)
    # Add statistics annotations (median values)
    # y_max = ax.get_ylim()[1]
    # for i, (data, label) in enumerate(zip(box_data, box_labels)):
    #     if len(data) > 0:
    #         median_val = np.median(data)
    #         # Add median value as text above each box
    #         ax.text(positions[i], median_val - (ax.get_ylim()[1] * 0.01), f'{median_val:.0f}ms', 
    #            ha='center', va='bottom', fontsize=9, fontweight='bold',
    #            bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=1, edgecolor='none'))
    
    plt.tight_layout()
    plt.minorticks_off()
    
    if save_plots:
        save_plot(fig, f'{out_dir}/comparative_latency_distribution')
        #plt.savefig(f'{out_dir}/comparative_latency_distribution.png', dpi=600, bbox_inches='tight')
    
    plt.ion()

    # Print summary statistics
    print("\n" + "="*80)
    print("COMPARATIVE LATENCY ANALYSIS")
    print("="*80)
    
    for solution in solutions:
        if solution not in all_data:
            continue
            
        print(f"\n{solution.upper()}:")
        df = all_data[solution]
        
        for role in ['Regular', 'Malicious']:
            role_data = df[df['role'] == role]
            if len(role_data) > 0:
                latencies = role_data['duration_ms']
                print(f"  {role} Tenant:")
                print(f"    Requests: {len(latencies):,}")
                print(f"    Mean: {latencies.mean():.1f}ms")
                print(f"    Median: {latencies.median():.1f}ms")
                print(f"    P95: {latencies.quantile(0.95):.1f}ms")
                print(f"    P99: {latencies.quantile(0.99):.1f}ms")
                print(f"    Std Dev: {latencies.std():.1f}ms")

In [416]:
# plot all comparative latency distribution
plot_comparative_latency_distribution(
    data_dirs="./60_sec",
    save_plots=True,
    out_dir="./plots",
)

Loaded 105577 records for capsule
Loaded 35745 records for kubezoo
Loaded 361906 records for vcluster
Loaded 112940 records for kubevirt

COMPARATIVE LATENCY ANALYSIS

CAPSULE:
  Regular Tenant:
    Requests: 2,471
    Mean: 245.6ms
    Median: 96.0ms
    P95: 1402.0ms
    P99: 2718.4ms
    Std Dev: 512.1ms
  Malicious Tenant:
    Requests: 103,106
    Mean: 292.8ms
    Median: 131.0ms
    P95: 1329.0ms
    P99: 2792.9ms
    Std Dev: 529.0ms

KUBEZOO:
  Regular Tenant:
    Requests: 673
    Mean: 906.3ms
    Median: 1018.0ms
    P95: 1531.0ms
    P99: 2549.0ms
    Std Dev: 480.0ms
  Malicious Tenant:
    Requests: 35,072
    Mean: 866.0ms
    Median: 1018.0ms
    P95: 1531.0ms
    P99: 1532.0ms
    Std Dev: 364.2ms

VCLUSTER:
  Regular Tenant:
    Requests: 6,655
    Mean: 89.7ms
    Median: 85.0ms
    P95: 159.0ms
    P99: 196.0ms
    Std Dev: 37.9ms
  Malicious Tenant:
    Requests: 355,251
    Mean: 83.6ms
    Median: 81.0ms
    P95: 152.0ms
    P99: 188.0ms
    Std Dev: 40.4ms

KUB

## Plot the data using Capsule

In [376]:
# Run the analysis
analyze_fairness_test("./60_sec/capsule/data", "./60_sec/capsule/results")

Loaded 5007 baseline request records
Loaded 105577 stress test request records
FAIRNESS TEST SUMMARY - INDIVIDUAL REQUEST ANALYSIS
Test Result: ✅ PASSED

Test Configuration:
  Regular Requesters: 10
  Malicious Requesters: 500
  Regular Request Rate: 50.0/s
  Malicious Request Rate: 5000.0/s
  Test Duration: 60s

Individual Request Statistics:

  Malicious Tenant:
    Total Requests: 103,106
    Average Latency: 292.8ms
    Median Latency: 131.0ms
    95th Percentile: 1329.0ms
    Standard Deviation: 529.0ms
    Error Rate: 0.00%
    Successful Requests: 103,106
    Failed Requests: 0

  Regular Tenant:
    Total Requests: 2,471
    Average Latency: 245.6ms
    Median Latency: 96.0ms
    95th Percentile: 1402.0ms
    Standard Deviation: 512.1ms
    Error Rate: 0.24%
    Successful Requests: 2,465
    Failed Requests: 6

Baseline Comparison:
  Baseline Latency: 8.0ms
  Current Avg Latency: 245.6ms
  Relative Increase: 2970.5%

Generating baseline vs stress comparison plot...
Plotting co

(      start_time     role  duration_ms operation    resource  is_error  \
 0       0.002193  Regular           30    Create  Deployment     False   
 1       0.002195  Regular           16    Create  Deployment     False   
 2       0.002228  Regular           64    Create  Deployment     False   
 3       0.002238  Regular           52    Create  Deployment     False   
 4       0.002254  Regular           29    Create  Deployment     False   
 ...          ...      ...          ...       ...         ...       ...   
 5002   10.002069  Regular            7    Create  Deployment     False   
 5003   10.002103  Regular            6    Create  Deployment     False   
 5004   10.002108  Regular            7    Create  Deployment     False   
 5005   10.002112  Regular            7    Create  Deployment     False   
 5006   10.002162  Regular            7    Create  Deployment     False   
 
       elapsed_seconds  duration_seconds     phase  
 0            0.000000             0.030  bas

In [240]:
# Run the analysis
analyze_fairness_test("./60_sec/capsule/data", "./60_sec/capsule/results")

Loaded 5007 baseline request records
Loaded 105577 stress test request records
FAIRNESS TEST SUMMARY - INDIVIDUAL REQUEST ANALYSIS
Test Result: ✅ PASSED

Test Configuration:
  Regular Requesters: 10
  Malicious Requesters: 500
  Regular Request Rate: 50.0/s
  Malicious Request Rate: 5000.0/s
  Test Duration: 60s

Individual Request Statistics:

  Malicious Tenant:
    Total Requests: 103,106
    Average Latency: 292.8ms
    Median Latency: 131.0ms
    95th Percentile: 1329.0ms
    Standard Deviation: 529.0ms
    Error Rate: 0.00%
    Successful Requests: 103,106
    Failed Requests: 0

  Regular Tenant:
    Total Requests: 2,471
    Average Latency: 245.6ms
    Median Latency: 96.0ms
    95th Percentile: 1402.0ms
    Standard Deviation: 512.1ms
    Error Rate: 0.24%
    Successful Requests: 2,465
    Failed Requests: 6

Baseline Comparison:
  Baseline Latency: 8.0ms
  Current Avg Latency: 245.6ms
  Relative Increase: 2970.5%

Generating baseline vs stress comparison plot...
Plotting co

Traceback (most recent call last):
  File "/tmp/ipykernel_25115/1504283947.py", line 724, in analyze_fairness_test
    plot_individual_requests_scatter(df_stress, metadata, stress_timestamp, save_plots=True, out_dir=out_dir)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_25115/1504283947.py", line 127, in plot_individual_requests_scatter
    plt.savefig(f'{out_dir}/individual_requests_{timestamp}.png', dpi=300, bbox_inches='tight')
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/attilio/.local/lib/python3.13/site-packages/matplotlib/pyplot.py", line 1251, in savefig
    res = fig.savefig(*args, **kwargs)  # type: ignore[func-returns-value]
  File "/home/attilio/.local/lib/python3.13/site-packages/matplotlib/figure.py", line 3490, in savefig
    self.canvas.print_figure(fname, **kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/

(None, None)

## Plot the data using vCluster

In [241]:
# Run the analysis
df, metadata = analyze_fairness_test("./60_sec/vcluster/data", "./60_sec/vcluster/results")

Error during analysis: No baseline test data files found


Traceback (most recent call last):
  File "/tmp/ipykernel_25115/1504283947.py", line 703, in analyze_fairness_test
    df_baseline, baseline_metadata, baseline_timestamp = load_baseline_test_data(src_dir)
                                                         ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_25115/1224682815.py", line 52, in load_baseline_test_data
    raise FileNotFoundError("No baseline test data files found")
FileNotFoundError: No baseline test data files found


## Plot the data using KubeZoo

In [242]:
# Run the analysis
df, metadata = analyze_fairness_test("./60_sec/kubezoo/data", "./60_sec/kubezoo/results")

Error during analysis: No baseline test data files found


Traceback (most recent call last):
  File "/tmp/ipykernel_25115/1504283947.py", line 703, in analyze_fairness_test
    df_baseline, baseline_metadata, baseline_timestamp = load_baseline_test_data(src_dir)
                                                         ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_25115/1224682815.py", line 52, in load_baseline_test_data
    raise FileNotFoundError("No baseline test data files found")
FileNotFoundError: No baseline test data files found


In [243]:
# Run the analysis
df, metadata = analyze_fairness_test("./60_sec/kubevirt/data", "./60_sec/kubevirt/results")

Error during analysis: No baseline test data files found


Traceback (most recent call last):
  File "/tmp/ipykernel_25115/1504283947.py", line 703, in analyze_fairness_test
    df_baseline, baseline_metadata, baseline_timestamp = load_baseline_test_data(src_dir)
                                                         ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_25115/1224682815.py", line 52, in load_baseline_test_data
    raise FileNotFoundError("No baseline test data files found")
FileNotFoundError: No baseline test data files found
